# Mapas Tendencias Niveles
***
Esté código busca representar las tendencias medias por cuencas y por niveles usando la máscara creada en la carpeta Crea Cuencas. Para ello se usan los ficheros tendecy_levels obtenidos en calcula tendencia niveles y se promedia las tendencias por cuencas. Para ellos los pasos a seguir son los siguientes:
1. Abrir el fichero con las tendencias
2. Calcular el promedio por máscara y nivel
3. Representar

In [1]:
import numpy as np
import xarray as xr
import pandas as pd
from scipy.stats import median_abs_deviation

import matplotlib.pyplot as plt
import cartopy
import cartopy.feature as cfeature
import cartopy.crs as ccrs
from matplotlib.colors import ListedColormap, BoundaryNorm

from tqdm import tqdm

In [2]:
level_step = 100
resolution = '25'
dates = '1990_2025'
pressure_max = None
pressure_min = '4000'
pressure_code = pressure_min[0] + 'k' # Poner + pressure_max[0] si hace falta
robust = True
median = False
if robust:
    file_name = f'tendency_levels_{dates}_{resolution}_{pressure_code}_robust'
    fig_name = f'{dates}_{resolution}_{pressure_code}_robust'
else :
    file_name = f'tendency_levels_{dates}_{resolution}_{pressure_code}'
    fig_name = f'{dates}_{resolution}'

### Apertura de los datos de tendencia y estructura

In [3]:
ds = xr.open_dataset(f'./Data/tendency_levels/{file_name}.nc')
ds

<xarray.Dataset> Size: 769MB
Dimensions:    (latitude: 563, longitude: 1355, pressure: 125)
Coordinates:
  * latitude   (latitude) float64 5kB -77.75 -77.5 -77.25 ... 62.75 63.0 63.25
  * longitude  (longitude) float64 11kB -180.0 -179.8 -179.5 ... 179.8 180.0
    mask       (latitude, longitude) float64 6MB ...
  * pressure   (pressure) int64 1kB 4000 4020 4040 4060 ... 6420 6440 6460 6480
Data variables:
    tendency   (latitude, longitude, pressure) float64 763MB ...
Attributes:
    description:  Dataset of temperature tendencies by pressure levels betwee...

Empezamos calculando la media y desviación por cuencas

In [6]:
def Mapas_medias(ds, press_range, median = False):
    # Importing the unique values of the mask
    cuencas = np.unique(ds.mask.values)

    basin_idx = [] # Number of the basin
    basin_mean = [] # Tendency mean asociated
    basin_std = [] # Tendency std asociated

    # Iterate for every 'cuenca'
    for cuenca in cuencas:
        if np.isnan(cuenca): # Skip if cuenca is Nan
            continue

        else:
            tendency = ds.tendency.where(ds.mask == cuenca).values
            if median:
                tendency_mean = np.nanmedian(tendency)
                tendency_std = median_abs_deviation(tendency, axis = None, nan_policy = 'omit')
            else:
                tendency_mean = np.nanmean(tendency)
                tendency_std = np.nanstd(tendency)
            basin_idx.append(cuenca)
            basin_mean.append(tendency_mean)
            basin_std.append(tendency_std)
    
    # Load the values of lat, lon and mask
    latitudes = ds.latitude.values
    longitudes = ds.longitude.values
    mask = ds.mask.values

    # Creating empty matrix
    tendency_mean = np.full((len(latitudes), len(longitudes)), np.nan)
    tendency_std = np.full((len(latitudes), len(longitudes)), np.nan)

    # Iterate for each basin
    for k, basin in enumerate(basin_idx):
        tendency_mean = np.where(mask == basin, basin_mean[k], tendency_mean)
        tendency_std = np.where(mask == basin, basin_std[k], tendency_std)

    # Creating figure
    fig, ax = plt.subplots(figsize=(20,10),
    subplot_kw={'projection': ccrs.InterruptedGoodeHomolosine(central_longitude=-160,emphasis='ocean')})

    # Ploting mean
    means = ax.pcolormesh(longitudes, latitudes, tendency_mean, cmap = "jet", transform = ccrs.PlateCarree())

    # Colorbar
    fig.colorbar(mappable = means, shrink = 0.8, label = 'Mean Tendency ºC / century')

    # Plotting ocupations
    #ds.n.plot(ax = ax, transform = ccrs.PlateCarree(), add_colorbar = False)

    # Plotting features
    ax.set_global()
    ax.coastlines(resolution='110m')
    ax.add_feature(cfeature.LAND, facecolor='lightgray')

    # Title
    ax.set_title(f'Mean tendency for {press_range} dbars in {dates} (Resolution: {resolution})')

    # Plotting text
    for k, basin in enumerate(basin_idx):
        if not np.isnan(basin_mean[k]):
            latitude = ds.latitude.where(ds.mask == basin).values
            longitude = ds.longitude.where(ds.mask == basin).values

            # Normal mean for latitude
            mean_lat = np.nanmean(latitude)

            # Ciruclar mean for longitude
            lon_rad = np.radians(longitude)
            mean_lon = np.degrees(np.arctan2(np.nanmean(np.sin(lon_rad)), np.nanmean(np.cos(lon_rad))))

            ax.text(mean_lon,mean_lat, rf'{basin_mean[k]:.3f}$\pm${basin_std[k]:.3f}', transform=ccrs.PlateCarree(),fontsize=12,ha='center',va='center',bbox=dict(facecolor='white', alpha=0.8,edgecolor='none'))
        else:
            continue

    # Save plot
    if median:
        plt.savefig(f'./Data/plots/Mapas_Tendencias_levels/Mean_Tendency_{fig_name}_mean.png')
    else:
        plt.savefig(f'./Data/plots/Mapas_Tendencias_levels/Mean_Tendency_{fig_name}_{press_range}.png')

    plt.close()

In [7]:
presiones = np.arange(np.min(ds.pressure.values), np.max(ds.pressure.values), level_step)

for i in range(0, len(presiones)):
    ds_press = ds.sel(pressure=presiones[i])
    press_range = str(presiones[i]).split()[0] + 'k'
    Mapas_medias(ds_press, press_range)

C:\Users\ismae\AppData\Local\Temp\ipykernel_19964\3907781163.py:20: RuntimeWarning: Mean of empty slice
  tendency_mean = np.nanmean(tendency)
c:\Users\ismae\miniconda3\envs\practicas\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\ismae\AppData\Local\Temp\ipykernel_19964\3907781163.py:20: RuntimeWarning: Mean of empty slice
  tendency_mean = np.nanmean(tendency)
c:\Users\ismae\miniconda3\envs\practicas\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\ismae\AppData\Local\Temp\ipykernel_19964\3907781163.py:20: RuntimeWarning: Mean of empty slice
  tendency_mean = np.nanmean(tendency)
c:\Users\ismae\miniconda3\envs\practicas\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  v

In [8]:
ds.close()
ds_press.close()
plt.close('all')